# 00 - MuJoCo + TurtleBot3 setup check

Stage 0: confirm the ROBOTIS TurtleBot3 model loads, the wheels drive as expected, and offscreen rendering works in Colab. Model source: ROBOTIS `robotis_mujoco_menagerie` (Apache-2.0). Nothing here is part of the final pipeline yet.

In [ ]:
# Install MuJoCo and fetch only the TurtleBot3 folder of the ROBOTIS model repository
!pip install -q mujoco
!test -d robotis_mujoco_menagerie || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ROBOTIS-GIT/robotis_mujoco_menagerie.git
!cd robotis_mujoco_menagerie && git sparse-checkout set robotis_tb3
!ls robotis_mujoco_menagerie/robotis_tb3

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # offscreen rendering; must be set before importing mujoco
import numpy as np
import matplotlib.pyplot as plt
import mujoco

MODEL = "waffle_pi"   # "burger" or "waffle_pi"
WHEEL_RADIUS = 0.033  # m, wheel body height in the XML
WHEEL_SEP = {"burger": 0.16, "waffle_pi": 0.288}[MODEL]   # m, twice the wheel body y offset in the XML

model = mujoco.MjModel.from_xml_path(f"robotis_mujoco_menagerie/robotis_tb3/scene_turtlebot3_{MODEL}.xml")
data = mujoco.MjData(model)
print("timestep", model.opt.timestep, "| bodies", model.nbody, "| actuators", model.nu)
for i in range(model.nu):
    print("actuator", model.actuator(i).name, "ctrlrange", model.actuator_ctrlrange[i])

In [ ]:
def yaw_of(d):
    w, x, y, z = d.qpos[3:7]   # free-joint quaternion order is (w, x, y, z)
    return np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))

def run(ctrl_left, ctrl_right, seconds, settle=0.5):
    mujoco.mj_resetData(model, data)
    data.ctrl[:] = 0
    mujoco.mj_forward(model, data)
    for _ in range(int(settle / model.opt.timestep)):
        mujoco.mj_step(model, data)
    x0, y0, yaw0 = data.qpos[0], data.qpos[1], yaw_of(data)
    data.ctrl[model.actuator("wheel_left").id] = ctrl_left
    data.ctrl[model.actuator("wheel_right").id] = ctrl_right
    for _ in range(int(seconds / model.opt.timestep)):
        mujoco.mj_step(model, data)
    dist = np.hypot(data.qpos[0] - x0, data.qpos[1] - y0)
    dyaw = (yaw_of(data) - yaw0 + np.pi) % (2 * np.pi) - np.pi
    wl = data.joint("wheel_left").qvel[0]
    wr = data.joint("wheel_right").qvel[0]
    return dist / seconds, dyaw / seconds, wl, wr

CMD = 3.0   # rad/s wheel velocity command

v, wz, wl, wr = run(CMD, CMD, 3.0)
print(f"straight: command {CMD} rad/s -> measured wheel speeds L {wl:.2f}, R {wr:.2f} rad/s")
print(f"          mean forward speed {v:.3f} m/s (ideal r*cmd = {WHEEL_RADIUS * CMD:.3f}), yaw rate {wz:+.3f} rad/s (ideal 0)")

v, wz, wl, wr = run(-CMD, CMD, 2.0)
print(f"spin:     command L {-CMD}, R {CMD} rad/s -> measured wheel speeds L {wl:.2f}, R {wr:.2f} rad/s")
print(f"          yaw rate {wz:+.3f} rad/s (ideal r*(wR-wL)/sep = {WHEEL_RADIUS * 2 * CMD / WHEEL_SEP:.3f})")
print(f"          effective wheel separation from the measured yaw rate: {WHEEL_RADIUS * (wr - wl) / wz:.3f} m (XML value {WHEEL_SEP} m)")

In [ ]:
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)
try:
    renderer = mujoco.Renderer(model, height=480, width=640)
    renderer.update_scene(data)
    plt.imshow(renderer.render())
    plt.axis("off")
    plt.show()
except Exception as e:
    print("Offscreen rendering failed:", e)
    print("Try MUJOCO_GL=osmesa after: !apt-get install -y libosmesa6-dev")

## Steady-state drive characterisation (Waffle Pi, unmodified upstream XML)

Statistics are taken over the last second of each 4 s run so the spin-up transient is excluded. Observed in one run (to be re-checked, not yet a calibrated model):

- Wheel speed is roughly command minus 2.04 rad/s for commands above that, and the wheels do not move below it.
- Signed forward speed over r times mean wheel speed: about 1.0 at command 3.0, about 0.86 at commands 5.0 and 7.88.
- In-place spin gives about 74 to 76 percent of the yaw rate predicted from measured wheel speeds and the XML wheel separation (effective separation about 0.38 to 0.39 m).

In [ ]:
def run_log(ctrl_left, ctrl_right, seconds=4.0, settle=0.5):
    mujoco.mj_resetData(model, data)
    data.ctrl[:] = 0
    mujoco.mj_forward(model, data)
    for _ in range(int(settle / model.opt.timestep)):
        mujoco.mj_step(model, data)
    data.ctrl[model.actuator("wheel_left").id] = ctrl_left
    data.ctrl[model.actuator("wheel_right").id] = ctrl_right
    log = []
    for _ in range(int(seconds / model.opt.timestep)):
        mujoco.mj_step(model, data)
        log.append([data.time, data.qpos[0], data.qpos[1], yaw_of(data),
                    data.joint("wheel_left").qvel[0], data.joint("wheel_right").qvel[0]])
    return np.array(log)

def steady(log, window=1.0):
    # Statistics over the last `window` seconds only, so the spin-up transient is excluded
    n = int(window / model.opt.timestep)
    s = log[-n:]
    dt = s[-1, 0] - s[0, 0]
    heading = s[:, 3].mean()
    v = ((s[-1, 1] - s[0, 1]) * np.cos(heading) + (s[-1, 2] - s[0, 2]) * np.sin(heading)) / dt   # signed forward speed
    wz = (np.unwrap(s[:, 3])[-1] - np.unwrap(s[:, 3])[0]) / dt
    return v, wz, s[:, 4].mean(), s[:, 5].mean()

print("straight: cmd | wheel L, R  | signed v | r*mean(wheel) | yaw rate")
for cmd in (1.0, 2.0, 3.0, 5.0, 7.88):
    v, wz, wl, wr = steady(run_log(cmd, cmd))
    print(f"          {cmd:5.2f} | {wl:6.2f} {wr:6.2f} | {v:+.3f} | {WHEEL_RADIUS * (wl + wr) / 2:.3f} | {wz:+.3f}")

print("spin:     cmd | wheel L, R  | yaw rate | ideal from measured wheels")
for cmd in (1.0, 3.0, 7.88):
    v, wz, wl, wr = steady(run_log(-cmd, cmd))
    print(f"          {cmd:5.2f} | {wl:6.2f} {wr:6.2f} | {wz:+.3f} | {WHEEL_RADIUS * (wr - wl) / WHEEL_SEP:+.3f}")